# Power Transforms — Box-Cox and Yeo-Johnson

Where the previous notebook tried hand-picked transforms (`log1p`, `sqrt`, ...), `PowerTransformer` finds the **best-fitting power transform automatically** for each column, by searching for the optimal parameter (called **lambda, λ**) that makes the data as close to normal as possible. Two variants are compared here: **Box-Cox** (needs strictly positive data) and **Yeo-Johnson** (works with zero/negative values too).

## 1. Setup and Data

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import scipy.stats as stats

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LinearRegression

from sklearn.metrics import r2_score

from sklearn.preprocessing import PowerTransformer

In [3]:
df = pd.read_csv('concrete_data.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'concrete_data.csv'

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
X = df.drop(columns=['Strength'])
y = df.iloc[:,-1]

## 2. Baseline — Linear Regression Without Any Transform

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

### A More Reliable Estimate — Cross-Validation

A single `train_test_split()` R² score can shift a fair amount depending on which rows land in the test set — averaging across 5 folds gives a steadier baseline to compare transforms against.

In [ ]:
# Applying Regression without any transformation
lr = LinearRegression()

lr.fit(X_train,y_train)

y_pred = lr.predict(X_test)

r2_score(y_test,y_pred)

## 3. Checking Each Feature's Distribution

Same pairing as the log-transform notebook: a distribution plot alongside a Q-Q plot for every feature, to see which ones are skewed enough to be worth transforming at all.

### ⚠️ `sns.distplot()` Is Removed on Current Seaborn

Replaced with `sns.histplot(kde=True)` everywhere it appears in this notebook.

In [ ]:
for col in X_train.columns:
    plt.figure(figsize=(14, 4))
    plt.subplot(121)
    sns.histplot(X_train[col], kde=True)
    plt.title(col)

    plt.subplot(122)
    stats.probplot(X_train[col], dist='norm', plot=plt)
    plt.title(col)

    plt.show()


## 4. Box-Cox Transform

Box-Cox searches for the power `λ` that best normalizes each column's distribution. It has one hard requirement: **every value must be strictly positive** — the underlying formula involves `log(x)` when λ=0, which is undefined at `x ≤ 0`. That's exactly why `+ 0.000001` is added below: it nudges any exact-zero values in the concrete dataset just above zero, without meaningfully changing anything else.

In [ ]:
# Applying Box-Cox Transform

pt = PowerTransformer(method='box-cox')

X_train_transformed = pt.fit_transform(X_train+0.000001)
X_test_transformed = pt.transform(X_test+0.000001)

pd.DataFrame({'cols':X_train.columns,'box_cox_lambdas':pt.lambdas_})

### Reading the Lambda Values

Each column gets its **own** λ, chosen independently to best-normalize that specific column. A few reference points for interpreting the numbers above:
- **λ = 1** → essentially no transform needed (data was already close to normal)
- **λ = 0.5** → roughly a square-root transform
- **λ = 0** → equivalent to a log transform
- **λ < 0** → a reciprocal-like transform (for heavily right-skewed data)

In [ ]:
# Applying linear regression on transformed data

lr = LinearRegression()
lr.fit(X_train_transformed,y_train)

y_pred2 = lr.predict(X_test_transformed)

r2_score(y_test,y_pred2)

### Cross-Validated Comparison

In [ ]:
# Using cross val score

pt = PowerTransformer(method='box-cox')
X_transformed = pt.fit_transform(X+0.0000001)

lr = LinearRegression()
np.mean(cross_val_score(lr,X_transformed,y,scoring='r2'))

In [ ]:
# Before and after comparision for Box-Cox Plot
X_train_transformed = pd.DataFrame(X_train_transformed,columns=X_train.columns)

for col in X_train_transformed.columns:
    plt.figure(figsize=(14,4))
    plt.subplot(121)
    sns.distplot(X_train[col])
    plt.title(col)

    plt.subplot(122)
    sns.distplot(X_train_transformed[col])
    plt.title(col)

    plt.show()

*(Original had a truncated `plt.title(` call cut off mid-loop — completed below.)*

In [ ]:
for col in X_train_transformed.columns:
    plt.figure(figsize=(14, 4))
    plt.subplot(121)
    sns.histplot(X_train[col], kde=True)
    plt.title(col + ' (before)')

    plt.subplot(122)
    sns.histplot(X_train_transformed[col], kde=True)
    plt.title(col + ' (after Box-Cox)')

    plt.show()


## 5. Yeo-Johnson Transform

`PowerTransformer()` with no `method=` argument defaults to **Yeo-Johnson** — a more flexible variant of the same idea that handles **zero and negative values natively**, with no need for the `+ 0.000001` workaround Box-Cox required.

In [ ]:
# Apply Yeo-Johnson transform

pt1 = PowerTransformer()

X_train_transformed2 = pt1.fit_transform(X_train)
X_test_transformed2 = pt1.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_transformed2,y_train)

y_pred3 = lr.predict(X_test_transformed2)

print(r2_score(y_test,y_pred3))

pd.DataFrame({'cols':X_train.columns,'Yeo_Johnson_lambdas':pt1.lambdas_})


In [ ]:
# applying cross val score

pt = PowerTransformer()
X_transformed2 = pt.fit_transform(X)

lr = LinearRegression()
np.mean(cross_val_score(lr,X_transformed2,y,scoring='r2'))

In [ ]:
X_train_transformed2 = pd.DataFrame(X_train_transformed2,columns=X_train.columns)

In [ ]:
# Before and after comparision for Yeo-Johnson

for col in X_train_transformed2.columns:
    plt.figure(figsize=(14,4))
    plt.subplot(121)
    sns.distplot(X_train[col])
    plt.title(col)

    plt.subplot(122)
    sns.distplot(X_train_transformed2[col])
    plt.title(col)

    plt.show()

In [ ]:
# Side by side Lambdas
pd.DataFrame({'cols':X_train.columns,'box_cox_lambdas':pt.lambdas_,'Yeo_Johnson_lambdas':pt1.lambdas_})

## 6. Putting All Three Results Side by Side

The notebook computed three separate R² scores along the way — worth pulling them into one table to see clearly whether either power transform actually beat the untransformed baseline, and by how much.

In [ ]:
comparison = pd.DataFrame({
    'Method': ['No transform', 'Box-Cox', 'Yeo-Johnson'],
    'Cross-validated R2': [
        np.mean(cross_val_score(LinearRegression(), X, y, scoring='r2')),
        np.mean(cross_val_score(LinearRegression(), pt.fit_transform(X + 0.0000001), y, scoring='r2')),
        np.mean(cross_val_score(LinearRegression(), pt1.fit_transform(X), y, scoring='r2')),
    ]
})
comparison


## Summary

| Task | Method |
|---|---|
| Auto-find the best power transform (positive data only) | `PowerTransformer(method='box-cox')` |
| Auto-find the best power transform (any data) | `PowerTransformer()` *(Yeo-Johnson, the default)* |
| See what transform was chosen per column | `pt.lambdas_` |
| Compare methods fairly | `cross_val_score(..., scoring='r2')`, averaged, not a single train/test split |

### Box-Cox vs. Yeo-Johnson — which to reach for
- All features strictly positive → either works; Box-Cox is the more "classic" choice.
- Any zeros or negative values present → **Yeo-Johnson only** (Box-Cox will error, or need an artificial offset like the `+0.000001` trick used above).
- Not sure a transform will even help? Check `.skew()` and a Q-Q plot first (from the previous notebook) — a column that's already close to normal doesn't need one.